In [1]:
# Standard library imports
import sys
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Cleaned Datasets

In [2]:
# Load cleaned datasets
# If cleaned data exists, load it; otherwise load from raw data

cleaned_crashes_path = Path('../data/processed/01_cleaned_crashes.parquet')
cleaned_persons_path = Path('../data/processed/02_cleaned_persons.parquet')

print("Loading datasets...")
print("=" * 60)

if cleaned_crashes_path.exists() and cleaned_persons_path.exists():
    print("Loading from cleaned processed data...")
    df_crashes = pd.read_parquet(cleaned_crashes_path)
    df_persons = pd.read_parquet(cleaned_persons_path)
else:
    raise FileNotFoundError(
        "Cleaned data not found! Please run notebooks 02 and 03 first to clean the data."
    )

print("\n📊 Dataset Overview:")
print(f"Crashes: {df_crashes.shape}")
print(f"Persons: {df_persons.shape}")

print(f"\nCrashes memory: {df_crashes.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Persons memory: {df_persons.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display first few rows
print("\n🔍 Crashes Sample:")
display(df_crashes.head(2))
print("\n🔍 Persons Sample:")
display(df_persons.head(2))

Loading datasets...
Loading from cleaned processed data...

📊 Dataset Overview:
Crashes: (2221559, 22)
Persons: (5600377, 16)

Crashes memory: 1506.77 MB
Persons memory: 3802.90 MB

🔍 Crashes Sample:


,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,NUMBER OF PEDESTRIANS INJURED,NUMBER OF PEDESTRIANS KILLED,NUMBER OF CYCLIST INJURED,NUMBER OF CYCLIST KILLED,NUMBER OF MOTORIST INJURED,NUMBER OF MOTORIST KILLED,CONTRIBUTING FACTOR VEHICLE 1,CONTRIBUTING FACTOR VEHICLE 2,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2
0,2021-09-11,2:39,Brooklyn,11207,40.720314,-73.92673,"(0.0, 0.0)",Whitestone Expressway,20 Avenue,2.0,0.0,0,0,0,0,2,0,Aggressive Driving/Road Rage,Unspecified,4455765,Sedan,Sedan
1,2022-03-26,11:45,Brooklyn,11207,40.720314,-73.92673,"(0.0, 0.0)",Queensboro Bridge Upper,3 Avenue,1.0,0.0,0,0,0,0,1,0,Pavement Slippery,Unspecified,4513547,Sedan,Sedan



🔍 Persons Sample:


,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,PERSON_ID,PERSON_TYPE,PERSON_INJURY,VEHICLE_ID,PERSON_AGE,EJECTION,EMOTIONAL_STATUS,BODILY_INJURY,POSITION_IN_VEHICLE,COMPLAINT,PED_ROLE,PERSON_SEX
0,10249006,4229554,2019-10-26,9:43,31aa2bc0-f545-444f-8cdb-f1cb5cf00b89,Occupant,Unspecified,19141108.0,36.0,Not Ejected,Does Not Apply,Does Not Apply,Driver,Does Not Apply,Registrant,U
1,10255054,4230587,2019-10-25,15:15,4629e500-a73e-48dc-b8fb-53124d124b80,Occupant,Unspecified,19144075.0,33.0,Not Ejected,Does Not Apply,Does Not Apply,"Front passenger, if two or more persons, inclu...",Does Not Apply,Passenger,F


In [3]:
# Select only essential columns to reduce memory footprint
print("\n🔧 Optimizing memory by selecting essential columns...")

# Essential crash columns
crash_cols_to_keep = [
    'COLLISION_ID', 'CRASH DATE', 'CRASH TIME', 'BOROUGH', 'ZIP CODE',
    'LATITUDE', 'LONGITUDE', 
    'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED',
    'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED',
    'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED',
    'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED',
    'CONTRIBUTING FACTOR VEHICLE 1', 'VEHICLE TYPE CODE 1'
]

# Filter to existing columns only
crash_cols = [col for col in crash_cols_to_keep if col in df_crashes.columns]
df_crashes_slim = df_crashes[crash_cols].copy()

# Essential person columns  
person_cols_to_keep = [
    'COLLISION_ID', 'PERSON_TYPE', 'PERSON_AGE', 'PERSON_SEX', 
    'PERSON_INJURY', 'SAFETY_EQUIPMENT', 'PED_ROLE', 
    'COMPLAINT', 'BODILY_INJURY', 'POSITION_IN_VEHICLE'
]

# Filter to existing columns only
person_cols = [col for col in person_cols_to_keep if col in df_persons.columns]
df_persons_slim = df_persons[person_cols].copy()

# Clear original dataframes
del df_crashes, df_persons
import gc
gc.collect()

print(f"✓ Memory optimization complete!")
print(f"  Crashes: {df_crashes_slim.shape} - {df_crashes_slim.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"  Persons: {df_persons_slim.shape} - {df_persons_slim.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"  Total: {(df_crashes_slim.memory_usage(deep=True).sum() + df_persons_slim.memory_usage(deep=True).sum()) / 1024**2:.2f} MB")

# Update variables for next cells
df_crashes = df_crashes_slim
df_persons = df_persons_slim
del df_crashes_slim, df_persons_slim
gc.collect()


🔧 Optimizing memory by selecting essential columns...
✓ Memory optimization complete!
  Crashes: (2221559, 17) - 836.16 MB
  Persons: (5600377, 9) - 2301.56 MB
  Total: 3137.72 MB


0

## 2. Memory Optimization: Select Essential Columns

**Problem**: Full datasets (5.3GB) exceed available memory for join operation.

**Solution**: Keep only essential columns needed for analysis before merging.

**Columns to Keep:**
- **Crashes**: COLLISION_ID, CRASH_DATE, CRASH_TIME, BOROUGH, ZIP_CODE, LATITUDE, LONGITUDE, NUMBER_OF_PERSONS_INJURED, NUMBER_OF_PERSONS_KILLED, NUMBER_OF_PEDESTRIANS_INJURED, NUMBER_OF_PEDESTRIANS_KILLED, NUMBER_OF_CYCLIST_INJURED, NUMBER_OF_CYCLIST_KILLED, NUMBER_OF_MOTORIST_INJURED, NUMBER_OF_MOTORIST_KILLED, CONTRIBUTING_FACTOR_VEHICLE_1, VEHICLE_TYPE_CODE_1
- **Persons**: COLLISION_ID, PERSON_TYPE, PERSON_AGE, PERSON_SEX, PERSON_INJURY, SAFETY_EQUIPMENT, PED_ROLE, COMPLAINT, BODILY_INJURY, POSITION_IN_VEHICLE

This reduces memory by ~60% while retaining all analysis-critical fields.

## 2. Integration Strategy

**Objective:** Integrate Crashes and Persons datasets using COLLISION_ID as the shared key.

**Why INNER JOIN?**
- Ensures only crashes with valid person records are included
- Prevents null-heavy rows from crashes without person data
- Maintains data quality and consistency
- Aligns with project goal: analyze crashes involving people

**Alternative Strategies Considered:**
- **LEFT JOIN**: Would keep all crashes, but create many null values for crashes without person records
- **RIGHT JOIN**: Would keep all persons, but lose crash-level context
- **OUTER JOIN**: Would create maximum null values from both sides

**Decision:** INNER JOIN provides the cleanest, most analysis-ready dataset.

In [4]:
# Perform integration using Polars with streaming engine
print("\n🔗 Performing INNER JOIN on COLLISION_ID...")
print("🚀 Using Polars streaming engine for memory-efficient processing...")

# Install polars if not available
try:
    import polars as pl
    print(f"✓ Polars version {pl.__version__} loaded")
except ImportError:
    print("Installing polars...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "polars"])
    import polars as pl
    print(f"✓ Polars version {pl.__version__} installed and loaded")

import gc
from pathlib import Path

print("\n1️⃣ Saving datasets to parquet for Polars scan...")
temp_dir = Path('../data/temp')
temp_dir.mkdir(parents=True, exist_ok=True)

crashes_temp = temp_dir / 'crashes_temp.parquet'
persons_temp = temp_dir / 'persons_temp.parquet'

df_crashes.to_parquet(crashes_temp, index=False)
df_persons.to_parquet(persons_temp, index=False)
print(f"   ✓ Saved temporary files")

# Free pandas memory
del df_crashes, df_persons
gc.collect()

print("\n2️⃣ Creating Polars LazyFrame scans...")
# Use scan_parquet for true lazy loading
lf_crashes = pl.scan_parquet(crashes_temp)
lf_persons = pl.scan_parquet(persons_temp)
print("   ✓ LazyFrames created with scan_parquet")

print("\n3️⃣ Planning streaming INNER JOIN...")
# Perform the join
lf_integrated = lf_crashes.join(
    lf_persons,
    on='COLLISION_ID',
    how='inner',
    suffix='_person'
)
print("   ✓ Join query planned")

print("\n4️⃣ Executing with streaming engine and saving to disk...")
# Execute with streaming engine and write directly to parquet
output_path = Path('../data/processed/integrated_raw.parquet')
output_path.parent.mkdir(parents=True, exist_ok=True)

# Use sink_parquet to write directly without loading into memory
lf_integrated.sink_parquet(str(output_path))
print(f"   ✓ Data streamed and saved to {output_path}")

print("\n5️⃣ Loading result (reading from disk)...")
# Read back a manageable portion for display
df_integrated = pd.read_parquet(output_path)

# Clean up temp files
crashes_temp.unlink()
persons_temp.unlink()
print("   ✓ Temporary files cleaned up")

gc.collect()

print("\n✅ Integration Complete!")
print(f"Integrated dataset: {df_integrated.shape}")
print(f"Memory usage: {df_integrated.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"File size: {output_path.stat().st_size / 1024**2:.2f} MB")
print(f"\nSample of merged data:")
display(df_integrated.head(3))


🔗 Performing INNER JOIN on COLLISION_ID...
🚀 Using Polars streaming engine for memory-efficient processing...
Installing polars...
✓ Polars version 1.35.2 installed and loaded

1️⃣ Saving datasets to parquet for Polars scan...
   ✓ Saved temporary files

2️⃣ Creating Polars LazyFrame scans...
   ✓ LazyFrames created with scan_parquet

3️⃣ Planning streaming INNER JOIN...
   ✓ Join query planned

4️⃣ Executing with streaming engine and saving to disk...
   ✓ Data streamed and saved to ..\data\processed\integrated_raw.parquet

5️⃣ Loading result (reading from disk)...
   ✓ Temporary files cleaned up

✅ Integration Complete!
Integrated dataset: (5600377, 25)
Memory usage: 4375.97 MB
File size: 91.82 MB

Sample of merged data:


,COLLISION_ID,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,NUMBER OF PEDESTRIANS INJURED,NUMBER OF PEDESTRIANS KILLED,NUMBER OF CYCLIST INJURED,NUMBER OF CYCLIST KILLED,NUMBER OF MOTORIST INJURED,NUMBER OF MOTORIST KILLED,CONTRIBUTING FACTOR VEHICLE 1,VEHICLE TYPE CODE 1,PERSON_TYPE,PERSON_AGE,PERSON_SEX,PERSON_INJURY,PED_ROLE,COMPLAINT,BODILY_INJURY,POSITION_IN_VEHICLE
0,3423492,2016-04-17,13:00,Queens,11378,40.720314,-73.92673,1.0,0.0,0,0,0,0,1,0,Driver Inattention/Distraction,4 Dr Sedan,Occupant,7.0,F,Unspecified,Passenger,Does Not Apply,Does Not Apply,Right rear passenger or motorcycle sidecar pas...
1,3895315,2018-05-05,20:00,Brooklyn,11207,40.720314,-73.92673,1.0,0.0,0,0,0,0,1,0,Driver Inattention/Distraction,Motorbike,Occupant,23.0,M,Injured,Driver,Complaint of Pain,Knee-Lower Leg Foot,Driver
2,3684260,2017-06-02,16:45,Brooklyn,11236,40.635777,-73.88665,0.0,0.0,0,0,0,0,0,0,Driver Inattention/Distraction,Bus,Occupant,6.0,F,Unspecified,Passenger,Does Not Apply,Does Not Apply,"Middle rear seat, or passenger lying across a ..."


## 3. Validate Integration

**Validation Checks:**
1. **Data Retention**: Verify minimal data loss during merge
2. **Missing Values**: Identify new missing values introduced by merge
3. **Relationships**: Confirm one-to-many relationship (crash → persons)
4. **Duplicates**: Check for duplicate rows
5. **Data Types**: Ensure types are preserved correctly

In [5]:
# Run comprehensive validation
print("\n" + "=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)

total_records = len(df_integrated)
unique_collisions = df_integrated['COLLISION_ID'].nunique()
avg_persons_per_crash = total_records / unique_collisions if unique_collisions > 0 else 0
duplicate_records = df_integrated.duplicated().sum()
memory_usage_mb = df_integrated.memory_usage(deep=True).sum() / (1024**2)

print(f"✓ Total records: {total_records:,}")
print(f"✓ Unique collisions: {unique_collisions:,}")
print(f"✓ Avg persons per crash: {avg_persons_per_crash:.2f}")
print(f"✓ Duplicate records: {duplicate_records:,}")
print(f"✓ Memory usage: {memory_usage_mb:.2f} MB")

# Show column overview
print(f"\n📋 Column Overview:")
print(f"Total columns: {df_integrated.shape[1]}")
print(f"\nSample columns:")
print(df_integrated.columns.tolist()[:20])


VALIDATION SUMMARY
✓ Total records: 5,600,377
✓ Unique collisions: 1,447,650
✓ Avg persons per crash: 3.87
✓ Duplicate records: 214,723
✓ Memory usage: 4375.97 MB

📋 Column Overview:
Total columns: 25

Sample columns:
['COLLISION_ID', 'CRASH DATE', 'CRASH TIME', 'BOROUGH', 'ZIP CODE', 'LATITUDE', 'LONGITUDE', 'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED', 'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED', 'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED', 'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED', 'CONTRIBUTING FACTOR VEHICLE 1', 'VEHICLE TYPE CODE 1', 'PERSON_TYPE', 'PERSON_AGE', 'PERSON_SEX']


## 4. Post-Integration Analysis

Before saving, let's examine the integrated dataset structure:

In [6]:
# Analyze the integrated dataset
print("=" * 60)
print("INTEGRATED DATASET ANALYSIS")
print("=" * 60)

print(f"\n📊 Dataset Statistics:")
print(f"  • Total records: {len(df_integrated):,}")
print(f"  • Unique collisions: {df_integrated['COLLISION_ID'].nunique():,}")

# Check for date columns (could be CRASH DATE or CRASH_DATE)
date_col = None
if 'CRASH DATE' in df_integrated.columns:
    date_col = 'CRASH DATE'
elif 'CRASH_DATE' in df_integrated.columns:
    date_col = 'CRASH_DATE'

if date_col:
    df_integrated[date_col] = pd.to_datetime(df_integrated[date_col], errors='coerce')
    print(f"  • Date range: {df_integrated[date_col].min()} to {df_integrated[date_col].max()}")

print(f"\n👥 Person-Level Statistics:")
if 'PERSON_TYPE' in df_integrated.columns:
    print(f"  • Person types:")
    print(df_integrated['PERSON_TYPE'].value_counts().head())

print(f"\n🗺️ Geographic Coverage:")
if 'BOROUGH' in df_integrated.columns:
    print(f"  • Crashes by borough:")
    print(df_integrated['BOROUGH'].value_counts())

# Check for issues that need post-integration cleaning
print(f"\n⚠️  Issues Requiring Post-Integration Cleaning:")
missing_cols = df_integrated.isnull().sum()
problematic_cols = missing_cols[missing_cols > 0].sort_values(ascending=False).head(10)
if len(problematic_cols) > 0:
    print(f"  • Columns with missing values:")
    for col, count in problematic_cols.items():
        pct = (count / len(df_integrated)) * 100
        print(f"    - {col}: {count:,} ({pct:.1f}%)")
else:
    print(f"  ✓ No missing values detected")

INTEGRATED DATASET ANALYSIS

📊 Dataset Statistics:
  • Total records: 5,600,377
  • Unique collisions: 1,447,650
  • Date range: 2012-07-18 00:00:00 to 2025-11-16 00:00:00

👥 Person-Level Statistics:
  • Person types:
PERSON_TYPE
Occupant           5425820
Pedestrian           99250
Bicyclist            63092
Other Motorized      12215
Name: count, dtype: int64

🗺️ Geographic Coverage:
  • Crashes by borough:
BOROUGH
Brooklyn         3247302
Queens            950730
Manhattan         697567
Bronx             568076
Staten Island     136702
Name: count, dtype: int64

⚠️  Issues Requiring Post-Integration Cleaning:
  ✓ No missing values detected


## 5. Save Integrated Data

**Output Format:** Parquet (efficient, preserves data types, compresses well)

In [ ]:
# Save integrated dataset
output_path = Path('../data/processed/integrated_raw.parquet')
output_path.parent.mkdir(parents=True, exist_ok=True)

df_integrated.to_parquet(output_path, index=False, compression='snappy')

print(f"✅ Integrated dataset saved successfully!")
print(f"   Path: {output_path}")
print(f"   Size: {output_path.stat().st_size / (1024**2):.2f} MB")
print(f"   Records: {len(df_integrated):,}")

print("\n" + "=" * 60)
print("NEXT STEPS")
print("=" * 60)
print("1. ✓ Datasets integrated using INNER JOIN")
print("2. ✓ Validation checks completed")
print("3. ✓ Raw integrated data saved")
print("4. → NEXT: Run notebook 05_post_integration_cleaning.ipynb")
print("   This will handle:")
print("   - Remove redundant columns")
print("   - Clean new missing values")
print("   - Fix data type inconsistencies")
print("   - Standardize formats")
print("   - Create final clean dataset")

✅ Integrated dataset saved successfully!
   Path: ..\data\processed\integrated_raw.parquet
   Size: 96.23 MB
   Records: 5,600,377

NEXT STEPS
1. ✓ Datasets integrated using INNER JOIN
2. ✓ Validation checks completed
3. ✓ Raw integrated data saved
4. → NEXT: Run notebook 05_post_integration_cleaning.ipynb
   This will handle:
   - Remove redundant columns
   - Clean new missing values
   - Fix data type inconsistencies
   - Standardize formats
   - Create final clean dataset


: 